In [1]:
# Imports
import matplotlib.pyplot as plt
import numpy as np
import matplotlib as mpl

In [4]:
# Constants
regions = ['Great Britain', 'Germany', 'California', 'Texas', 'South Africa', 'Tokyo', 'New South Wales']
short_regions = ['gb', 'de', 'ca', 'tx', 'zaf', 'tyo', 'nsw']
months = ['January', 'February', 'March', 'April', 'May', 'June', 'July', 'August', 'September', 'October', 'November', 'December']
short_months = ['jan', 'feb', 'mar', 'apr', 'may', 'jun', 'jul', 'aug', 'sep', 'oct', 'nov', 'dec'] 
month_dates = ['08012024-13012024', '12022024-17022024', '11032024-16032024',  
    '08042024-13042024', '13052024-18052024', '10062024-15062024',  
    '08072024-13072024', '12082024-17082024', '09092024-14092024',  
    '14102024-19102024', '11112024-16112024', '09122024-14122024']
region_folders = [f'../data/results/entire-wf-shifting/out-err/{region}/' for region in short_regions]
workflows=['chipseq', 'mag', 'montage', 'nanoseq', 'rangeland', 'rnaseq', 'sarek']

In [5]:
# Parse Explorer Summary (NEW)
def parse_explorer_summary(explorer_data):
    with open(explorer_data, 'r') as f:
        data = [line.strip().split(',') for line in f.readlines()]

    for row in data:
        if '~' in row[0]:
            shift_ms = row[0].split('~')[1][:-4]
            days = int(shift_ms.split('-')[0])
            hours = int(shift_ms.split('-')[1])
            shift_h = (days * 24) + hours
            row[0] = shift_h
        else:
            row[0] = 0

    data_d = {}

    for row in data:
        data_d[row[0]] = {'emissions': row[1]}

    return data_d


def get_minimum(data):
    minimum = float(data[0]['emissions'])
    shift = 0

    for key, entry in data.items():
        if float(entry['emissions']) < minimum:
            minimum = float(entry['emissions'])
            shift = key

    return (shift, minimum)


def get_reduction(original, new):
    orig = float(original)
    neww = float(new)
    return ((orig - neww) / abs(orig)) * 100


def overview(trace, print=False):
    data = parse_explorer_summary(trace)
    (min_shift, min_emissions) = get_minimum(data)

    if print:
        print(f"Original [0] CCF {data[0]['emissions']} gCO2e")  # original
        print(f"Minimum [{min_shift}] CCF {min_emissions} gCO2e")  # minimum
        print(f"Reduction {get_reduction(data[0]['emissions'], min_emissions):.2f}%")

    return (data, min_shift, min_emissions)


def average(one, two, three):
    return (float(one) + float(two) + float(three)) / 3

In [13]:
# Parse Readings for all readings, through the year
def parse_readings(window, marg='', err=''):
    region = 'gb'
    folder = f'../data/results/entire-wf-shifting/out-err/{region}/'

    if err == '':
        reduction_by_workflow = {}

        for i in range(0, 12):
            for workflow in workflows:
                path_one = f'{folder}explorer-{window}h-{workflow}-{short_months[i]}-1-{region}-{month_dates[i]}{err}{marg}~footprint.csv'
                path_two = f'{folder}explorer-{window}h-{workflow}-{short_months[i]}-2-{region}-{month_dates[i]}{err}{marg}~footprint.csv'
                path_three = f'{folder}explorer-{window}h-{workflow}-{short_months[i]}-3-{region}-{month_dates[i]}{err}{marg}~footprint.csv'

                (data_1, _, min_emissions_1) = overview(path_one)
                (data_2, _, min_emissions_2) = overview(path_two)
                (data_3, _, min_emissions_3) = overview(path_three)
                avg_orig_ems = average(data_1[0]['emissions'], data_2[0]['emissions'], data_3[0]['emissions'])
                avg_min_ems = average(min_emissions_1, min_emissions_2, min_emissions_3)
                avg_reduction = get_reduction(avg_orig_ems, avg_min_ems)

                if workflow in reduction_by_workflow:
                    reduction_by_workflow[workflow].append(avg_reduction)
                else:
                    reduction_by_workflow[workflow] = [avg_reduction]

        return reduction_by_workflow

    data = {}

    for rep in range(1, 6):
        reduction_by_workflow = {}

        for i in range(0, 12):
            for workflow in workflows:
                path_one = f'{folder}explorer-{window}h-{workflow}-{short_months[i]}-1-{region}-{month_dates[i]}{err}{marg}~footprint.csv'
                path_two = f'{folder}explorer-{window}h-{workflow}-{short_months[i]}-2-{region}-{month_dates[i]}{err}{marg}~footprint.csv'
                path_three = f'{folder}explorer-{window}h-{workflow}-{short_months[i]}-3-{region}-{month_dates[i]}{err}{marg}~footprint.csv'

                (data_1, _, min_emissions_1) = overview(path_one)
                (data_2, _, min_emissions_2) = overview(path_two)
                (data_3, _, min_emissions_3) = overview(path_three)
                avg_orig_ems = average(data_1[0]['emissions'], data_2[0]['emissions'], data_3[0]['emissions'])
                avg_min_ems = average(min_emissions_1, min_emissions_2, min_emissions_3)
                avg_reduction = get_reduction(avg_orig_ems, avg_min_ems)

                if workflow in reduction_by_workflow:
                    reduction_by_workflow[workflow].append(avg_reduction)
                else:
                    reduction_by_workflow[workflow] = [avg_reduction]
    
        data[rep] = reduction_by_workflow

    return data

In [ ]:
# Sensitivity Analysis -- CI

# Sensitivity Analysis -- Runtime

# Sensitivity Analysis -- Runtime + CI

# Upper + Lower Bound for Storage Costs during interrupted workflow shifting (scenario where we extend time spent)

In [ ]:
## Sensitivity Analysis - CI Experiments
region = 'gb'

# Average CI
readings_24h = parse_readings(window=24)
curr = []
for workflow in workflows:
    curr.append(readings_24h[region][workflow])
reads_24h_original = np.array(curr)
print(reads_24h_original)

# all_reads_24h_err5_reps = parse_readings(window=24,err='err5')
# average over 5 reps
# all_reads_24h_err10_reps = parse_readings(window=24,err='err10')
# average over 5 reps

# calculate error between the reduction possible for 5 and 10%

# explore averaging and significance

[[17.75578987 26.8447225  48.13600845 81.3306011  39.00014293 11.54882754
  70.29873057 64.83034003 39.07575381 49.45001385 28.35085008 11.45263473]
 [ 0.          0.         25.88876584 68.07598983  0.          4.35251372
  46.91015862 43.42778579 19.72835939 23.59119066  0.          0.        ]
 [24.02960204 56.74949242 61.5894858   0.         26.48966729  9.78006718
  45.82509937 35.96386284 38.55253478 56.17008541 22.4109009   0.        ]
 [14.69409644  2.63644332 44.93399092 81.54539111 24.85506712 17.46461824
  69.48715029 72.14901292 35.62292197 46.99047332  7.75008705  0.        ]
 [ 8.64578769  0.         37.38845726 79.05942385 15.83490337 14.81111747
  65.74961203 67.16443804 31.3242004  42.73964328  1.94215111  0.        ]
 [19.9282959  39.18361161 47.44977782 80.66965632 44.71828295 10.22107326
  70.41512107 60.8253055  38.75429188 54.41767086 37.39995256 24.79718423]
 [16.22477863 15.92248953 44.62259069 81.46217164 31.28377557 14.63854221
  70.1332101  68.74400246 35.676